In [61]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn

In [62]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,fractal_dimension_mean,radius_se,texture_se,perimeter_se,area_se,smoothness_se,compactness_se,concavity_se,concave points_se,symmetry_se,fractal_dimension_se,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,1.0950,0.9053,8.589,153.40,0.006399,0.04904,0.05373,0.01587,0.03003,0.006193,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,0.5435,0.7339,3.398,74.08,0.005225,0.01308,0.01860,0.01340,0.01389,0.003532,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,0.7456,0.7869,4.585,94.03,0.006150,0.04006,0.03832,0.02058,0.02250,0.004571,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,0.4956,1.1560,3.445,27.23,0.009110,0.07458,0.05661,0.01867,0.05963,0.009208,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,0.7572,0.7813,5.438,94.44,0.011490,0.02461,0.05688,0.01885,0.01756,0.005115,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [63]:
df.shape

(569, 33)

In [64]:
df.drop(columns=['id', 'Unnamed: 32'], inplace=True)

In [65]:
df.shape

(569, 31)

In [66]:
df.isna().sum()

,0
diagnosis,0
radius_mean,0
texture_mean,0
perimeter_mean,0
area_mean,0
smoothness_mean,0
compactness_mean,0
concavity_mean,0
concave points_mean,0
symmetry_mean,0


In [67]:
df.duplicated().sum()

np.int64(0)

In [68]:
X = df.drop(columns=['diagnosis'])
y = df['diagnosis']

In [69]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

In [70]:
X_train.shape

(455, 30)

In [71]:
y_train.shape

(455,)

In [72]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [73]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [74]:
X_train_tensor = torch.from_numpy(X_train.astype(np.float32))
X_test_tensor = torch.from_numpy(X_test.astype(np.float32))
y_train_tensor = torch.from_numpy(y_train.astype(np.float32))
y_test_tensor = torch.from_numpy(y_test.astype(np.float32))

In [75]:
X_train_tensor.shape

torch.Size([455, 30])

In [76]:
y_train_tensor.shape

torch.Size([455])

In [77]:
from torch.utils.data import Dataset, DataLoader

class CustomDataset(Dataset):

  def __init__(self, features, labels):
    self.features = features
    self.labels = labels

  def __len__(self):
    return self.features.shape[0]

  def __getitem__(self, index):
    return self.features[index], self.labels[index]

In [78]:
train_dataset = CustomDataset(X_train_tensor, y_train_tensor)
test_dataset = CustomDataset(y_train_tensor, y_test_tensor)

In [79]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

In [80]:
class Model(nn.Module):
  def __init__(self, num_features):
    super().__init__()
    self.linear = nn.Linear(num_features, 1)
    self.sigmoid = nn.Sigmoid()

  def forward(self, features):
    out = self.linear(features)
    out = self.sigmoid(out)
    return out

In [81]:
learning_rate = 0.1
epochs = 25

In [82]:
model = Model(X_train_tensor.shape[1])

optimizer = torch.optim.SGD(model.parameters(), lr = learning_rate)

loss_function = nn.BCELoss()

In [83]:
for epoch in range(epochs):

  for batch_features, batch_labels in train_loader:

    y_pred = model(batch_features)

    loss = loss_function(y_pred, batch_labels.view(-1,1))

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

  print(f'Epoch: {epoch + 1}, Loss: {loss.item()}')

Epoch: 1, Loss: 0.17452764511108398
Epoch: 2, Loss: 0.2011185884475708
Epoch: 3, Loss: 0.2577721178531647
Epoch: 4, Loss: 0.15974190831184387
Epoch: 5, Loss: 0.06087237596511841
Epoch: 6, Loss: 0.10201764851808548
Epoch: 7, Loss: 0.10157155245542526
Epoch: 8, Loss: 0.18075425922870636
Epoch: 9, Loss: 0.12154043465852737
Epoch: 10, Loss: 0.02175954356789589
Epoch: 11, Loss: 0.016271885484457016
Epoch: 12, Loss: 0.008721329271793365
Epoch: 13, Loss: 0.10796613246202469
Epoch: 14, Loss: 0.03145807608962059
Epoch: 15, Loss: 0.10079775750637054
Epoch: 16, Loss: 0.04674528166651726
Epoch: 17, Loss: 0.01889924891293049
Epoch: 18, Loss: 0.017639441415667534
Epoch: 19, Loss: 0.10352588444948196
Epoch: 20, Loss: 0.09787289798259735
Epoch: 21, Loss: 0.2549099624156952
Epoch: 22, Loss: 0.02147395722568035
Epoch: 23, Loss: 0.028198178857564926
Epoch: 24, Loss: 0.0021187409292906523
Epoch: 25, Loss: 0.029288718476891518
